# 11 — Controlled Go arena and candidate evaluation

**Purpose:** compare frozen Go players on paired games, retain every transition, and make conservative development promotion decisions. **Run All defaults to wiring validation, not a strength claim.** The normal board is 9×9 with the exact rules from notebook 01. No training, service publication or final/lockbox data access occurs here.

**Inputs:** notebook 00 recovery definitions, notebook 01 rules, injected player callables, and (for real strength runs) immutable checkpoint identities and a protocol fixed before play. Notebook 08 provides Go-native models; notebook 09 supplies self-play candidates; notebook 10 can provide GTP opponents.

**Outputs:** `reports/11_arena_validation.json`, a readable summary, and resumable artifacts under `checkpoints/arena11-*`. Default checks should take less than two minutes. A strength run takes roughly `2 × paired blocks × maximum plies × seconds per move`, plus verification/checkpoint overhead. Large runs require this notebook's successful exact-resume receipt.

**Recovery:** restart the kernel, Run All to load and validate definitions, then call `arena_run(PROJECT, SAME_RUN_ID, SAME_PROTOCOL, SAME_PLAYERS)`. Complete history, model parameters/buffers, all framework RNGs, player runtime state, active game, paired-block cursor and prior results are restored. Invalid manifests or changed sources/protocols/weights are rejected. The latest two valid snapshots and a best progress snapshot are retained; one writer holds the run lock. Frozen evaluation has no optimizer, scheduler or scaler, recorded explicitly as `None`. Model strength is never inferred from checkpoint progress.

Deadlines are observed around cooperative player calls: a stalled in-process player cannot be forcibly interrupted. Deadline violations are recorded as **UNKNOWN**, never invented losses. Move caps likewise yield UNKNOWN. Use a separately managed process adapter for untrusted or potentially hanging external engines.

**V2:** independently cross-check every applied move; preserve full chronological history for external references; summarize p50/p95/p99/worst move latency; run synthetic development tactics and predeclared multi-generation comparisons. Old v1 results remain archived and cannot be resumed with changed source. External reference engines are evaluation opponents only. No teaching labels or weights flow from them into AHR training.

In [ ]:
from pathlib import Path
import nbformat
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
assert (PROJECT / "AGENTS.md").is_file()
_boot = nbformat.read(PROJECT / "notebooks/00_instance_setup_and_recovery.ipynb", as_version=4)
for _cell in _boot.cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "00:ahr-export", "exec"), globals())
load_notebook_exports(PROJECT / "notebooks/01_go_rules_and_independent_verifier.ipynb", globals())
torch.set_num_threads(1)
print("Arena ready: only tagged definitions loaded; no upstream workflow executed.")

## Freeze a protocol before the first move

Each block uses one seeded opening and two games with candidate colours reversed. Both colours see the identical opening state, including superko history. Openings are generated without consulting either player, with no final-data reads. Seeds, opening moves, player identities, rules, budget and stopping rules enter the immutable protocol hash.

Promotion uses the mean of the two candidate game scores in each complete block and a conservative bounded-variable Hoeffding interval. This treats a **paired block** as the unit, rather than treating correlated colour-swapped games as independent. The confidence interpretation requires independently sampled opening seeds; repeated adaptive evaluations require separately allocated error budgets. A default 95% interval on each repeatedly tested candidate is not a family-wide guarantee. No optional stopping is used: all predeclared blocks must finish.

All blocks must complete, the lower confidence bound must exceed the fixed threshold, and there must be no illegal moves, deadline failures, work-budget violations or frozen-weight changes. Wiring fixtures and 3×3 tests cannot promote. Passing this gate means better performance against the named reference under this protocol; it does not establish global strength or solved Go.

In [ ]:
from dataclasses import dataclass, asdict, field
from typing import Callable
import math
import io
import inspect
import re
from collections import Counter

ARENA_SCHEMA = "ahr11-controlled-arena-v2"
ARENA_RESUME = "11: restart kernel, Run All, then arena_run with identical run_id/protocol/players"

def arena_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)

def arena_digest(value):
    return hashlib.sha256(arena_json(value).encode()).hexdigest()

def arena_sources(project, numbers=(0, 1, 11)):
    result = {}
    for number in numbers:
        matches = list((Path(project) / "notebooks").glob(f"{number:02d}_*.ipynb"))
        if len(matches) != 1:
            raise RuntimeError(f"Expected exactly one notebook {number:02d}")
        notebook = nbformat.read(matches[0], as_version=4)
        result[matches[0].name] = hashlib.sha256("\n\n".join(c.source for c in notebook.cells if c.cell_type == "code").encode()).hexdigest()
    return result

def arena_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {k: arena_cpu(v) for k, v in value.items()}
    if isinstance(value, list):
        return [arena_cpu(v) for v in value]
    if isinstance(value, tuple):
        return tuple(arena_cpu(v) for v in value)
    return copy.deepcopy(value)

def arena_tensor_digest(state):
    digest = hashlib.sha256()
    for key, value in sorted(state.items()):
        if not torch.is_tensor(value):
            raise ValueError("Model state must contain tensors")
        data = value.detach().cpu().contiguous()
        digest.update(arena_json([key, str(data.dtype), list(data.shape)]).encode())
        digest.update(data.reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()

@dataclass(frozen=True)
class ArenaProtocol:
    candidate: str = "candidate"
    opponent: str = "reference"
    seeds: tuple = (1103, 2207)
    size: int = 9
    komi: float = 7.5
    opening_plies: int = 2
    max_plies: int = 256
    time_budget_s: float = 0.5
    purpose: str = "wiring_smoke"
    alpha: float = 0.05
    promotion_threshold: float = 0.55
    minimum_blocks: int = 128
    work_budget: dict = field(default_factory=dict)
    error_budget_note: str = "Single predeclared candidate; allocate alpha across repeated tests."

    def __post_init__(self):
        if self.purpose not in ("wiring_smoke", "development_strength", "recovery_fixture"):
            raise ValueError("Arena supports development data only")
        if self.size not in (3, 9) or not math.isfinite(self.komi):
            raise ValueError("Use finite komi on a supported 3x3 or 9x9 board")
        if not self.candidate or not self.opponent or self.candidate == self.opponent:
            raise ValueError("Require two distinct named players")
        if not self.seeds or len(set(self.seeds)) != len(self.seeds) or any(type(s) is not int or s < 0 for s in self.seeds):
            raise ValueError("Require distinct nonnegative integer seeds")
        if type(self.opening_plies) is not int or type(self.max_plies) is not int or not 0 <= self.opening_plies < self.max_plies:
            raise ValueError("Invalid opening/move limits")
        if not math.isfinite(self.time_budget_s) or self.time_budget_s <= 0:
            raise ValueError("Require a positive finite thinking-time budget")
        if not 0 < self.alpha < 1 or not 0.5 <= self.promotion_threshold < 1 or self.minimum_blocks < 2:
            raise ValueError("Invalid confidence/promotion requirements")
        if set(self.work_budget) - {"max_nodes", "max_model_calls"} or any(type(v) is not int or v < 1 for v in self.work_budget.values()):
            raise ValueError("Supported work caps are positive max_nodes/max_model_calls")
        if self.purpose == "development_strength" and self.size != 9:
            raise ValueError("Strength promotion is a 9x9 protocol")

@dataclass
class FrozenArenaPlayer:
    name: str
    decide: Callable
    config: dict
    provenance: dict
    model: object = None
    get_runtime_state: Callable = None
    set_runtime_state: Callable = None
    prepare_position: Callable = None

    def __post_init__(self):
        if (self.get_runtime_state is None) != (self.set_runtime_state is None):
            raise ValueError("Stateful players require both runtime snapshot and restore hooks")
        if self.model is not None:
            self.model.eval()
            self.model.requires_grad_(False)
        self.initial_weights = self.weights_digest()
        arena_json(self.config)
        arena_json(self.provenance)

    def weights(self):
        return arena_cpu(self.model.state_dict()) if self.model is not None else {}

    def weights_digest(self):
        return arena_tensor_digest(self.weights())

    def identity(self):
        return {"name": self.name, "config": self.config, "provenance": self.provenance,
                "weights_sha256": self.initial_weights,
                "runtime": "stateful" if self.get_runtime_state else "stateless",
                "full_history_preparation": self.prepare_position is not None}

    def check_frozen(self):
        if self.weights_digest() != self.initial_weights:
            raise RuntimeError("Frozen arena weights changed: " + self.name)
        if self.model is not None and (self.model.training or any(p.requires_grad for p in self.model.parameters())):
            raise RuntimeError("Arena model left frozen inference mode")

    def runtime(self):
        return arena_cpu(self.get_runtime_state()) if self.get_runtime_state else None

    def restore(self, weights, runtime):
        if arena_tensor_digest(weights) != self.initial_weights:
            raise RuntimeError("Resume weights differ from declared player")
        if self.model is not None:
            self.model.load_state_dict(weights, strict=True)
        if self.set_runtime_state:
            self.set_runtime_state(runtime)
        elif runtime is not None:
            raise RuntimeError("Unexpected hidden player runtime state")
        self.check_frozen()

def arena_opening(protocol, seed):
    state = initial_state(protocol.size, protocol.komi)
    rng = random.Random(seed)
    actions = []
    for _ in range(protocol.opening_plies):
        choices = [a for a in legal_actions(state) if a != state.pass_action]
        if not choices:
            raise RuntimeError("Opening generation cannot fill requested number of legal non-pass plies")
        action = rng.choice(choices)
        state = play(state, action)
        actions.append(action)
    return {"seed": seed, "actions": actions, "state": to_dict(state)}

def arena_jobs(protocol):
    return [{"index": 2*i+j, "block": i, "seed": seed, "candidate_color": color}
            for i, seed in enumerate(protocol.seeds) for j, color in enumerate((BLACK, WHITE))]

def arena_identity(project, protocol, players):
    names = {protocol.candidate, protocol.opponent}
    if set(players) != names or any(players[name].name != name for name in names):
        raise ValueError("Player inventory differs from protocol")
    for player in players.values():
        player.check_frozen()
        if protocol.work_budget and player.config.get("work_budget") != protocol.work_budget:
            raise ValueError("Both players must declare the same fixed work budget")
    if protocol.purpose == "development_strength":
        candidate = players[protocol.candidate]
        if candidate.model is None or candidate.provenance.get("purpose") != "go_training" or candidate.provenance.get("trained_optimizer_steps", 0) < 1:
            raise ValueError("Strength candidate must be a trained, checkpoint-backed Go model")
        for player in players.values():
            if player.provenance.get("kind") == "fixture" or not player.provenance.get("checkpoint_sha256", player.provenance.get("binary_sha256")):
                raise ValueError("Strength opponents require immutable checkpoint or engine-binary identity")
    return json.loads(arena_json({"schema": ARENA_SCHEMA, "protocol": asdict(protocol), "rules": RULESET,
            "sources": arena_sources(project), "players": {name: p.identity() for name, p in sorted(players.items())},
            "openings": [arena_opening(protocol, seed) for seed in protocol.seeds]}))


## Verify transitions and checkpoint each event

The arena calls the rules implementation independently of player code. It never trusts a player's declared winner, legal mask or final score. Only an actual two-pass terminal state receives a game score. The initial seeded opening and every subsequent action are replayed when loading a checkpoint.

A checkpoint is committed at initialization, after every attempted ply, at each game boundary, and on completion. A run holds one filesystem lock throughout; malformed, missing, mismatched or corrupt state produces an explicit error. This is state recovery for evaluation. Wall-clock-limited search can legitimately make different choices on different hardware even when weights and RNGs are identical.

In [ ]:
def arena_strict_load(run_dir):
    run_dir = Path(run_dir)
    path = run_dir / "manifest.json"
    if not path.exists():
        if list(run_dir.glob("*.pt")):
            raise RuntimeError("Arena checkpoint manifest missing; refusing restart")
        return None
    try:
        manifest = json.loads(path.read_text())
        entries = manifest["entries"]
        if not entries or len({e["file"] for e in entries}) != len(entries):
            raise ValueError("Invalid manifest inventory")
        states = {}
        for entry in entries:
            if Path(entry["file"]).name != entry["file"]:
                raise ValueError("Unsafe manifest path")
            state = _checkpoint_entry_state(run_dir, entry)
            if state is None:
                raise ValueError("Invalid committed checkpoint hash/payload: " + entry["file"])
            states[entry["file"]] = state
        if manifest["latest"] not in states or manifest["best"] not in states:
            raise ValueError("Missing latest/best checkpoint")
        if states[manifest["latest"]]["global_step"] != max(s["global_step"] for s in states.values()):
            raise ValueError("Manifest latest cursor rewound")
        return states[manifest["latest"]]
    except (KeyError, TypeError, ValueError, OSError) as error:
        raise RuntimeError("Invalid arena manifest; refusing silent recovery/restart") from error

def arena_replay(identity, job, actions):
    protocol = ArenaProtocol(**identity["protocol"])
    opening = identity["openings"][job["block"]]
    board = initial_state(protocol.size, protocol.komi)
    for action in opening["actions"]:
        board = play(board, action)
    if to_dict(board) != opening["state"]:
        raise RuntimeError("Opening history mismatch")
    for action in actions:
        board = play(board, action)
    return board

def arena_validate_snapshot(snapshot, identity, *, verify_history=True):
    if not isinstance(snapshot, dict) or REQUIRED_CHECKPOINT_KEYS - set(snapshot):
        raise RuntimeError("Incomplete arena snapshot")
    if snapshot.get("schema") != ARENA_SCHEMA or snapshot["config"] != identity or snapshot.get("identity_sha256") != arena_digest(identity):
        raise RuntimeError("Arena protocol, source or player identity changed")
    if any(snapshot[key] is not None for key in ("optimizer", "scheduler", "scaler")):
        raise RuntimeError("Frozen evaluation cannot resume training state")
    if snapshot.get("optimizer_steps") != 0 or snapshot.get("replay") is not None or snapshot.get("distributed") != {"world_size": 1}:
        raise RuntimeError("Unexpected training/distributed state in frozen arena")
    protocol = ArenaProtocol(**identity["protocol"])
    jobs = arena_jobs(protocol)
    cursor = snapshot["sampler"].get("next_game")
    if type(cursor) is not int or not 0 <= cursor <= len(jobs) or len(snapshot["results"]) != cursor:
        raise RuntimeError("Arena cursor/results mismatch")
    if snapshot["complete"] != (cursor == len(jobs)) or type(snapshot["global_step"]) is not int or snapshot["global_step"] < 0:
        raise RuntimeError("Arena stage/completion mismatch")
    if set(snapshot["model"]) != set(identity["players"]) or set(snapshot["runtime"]) != set(identity["players"]) or set(snapshot["initial_runtime"]) != set(identity["players"]):
        raise RuntimeError("Arena model/runtime inventory mismatch")
    for name, weights in snapshot["model"].items():
        if arena_tensor_digest(weights) != identity["players"][name]["weights_sha256"]:
            raise RuntimeError("Arena model hash mismatch")
    for index, row in enumerate(snapshot["results"] if verify_history else []):
        if row["job"] != jobs[index]:
            raise RuntimeError("Result order differs from paired schedule")
        board = arena_replay(identity, row["job"], row["actions"])
        if to_dict(board) != row["final_state"]:
            raise RuntimeError("Result board/history mismatch")
        if row["status"] == "COMPLETE":
            score = area_score(board)
            candidate_score = 0.5 if score == 0 else float(score * row["job"]["candidate_color"] > 0)
            if not board.terminal or row["black_score"] != score or row["candidate_score"] != candidate_score:
                raise RuntimeError("Terminal result failed independent scoring")
        elif not row["status"].startswith("UNKNOWN_") or row["candidate_score"] is not None or row["black_score"] is not None:
            raise RuntimeError("Unknown/cutoff results must never be labelled losses")
    active = snapshot["active"]
    if active is not None:
        if cursor == len(jobs) or active["job"] != jobs[cursor]:
            raise RuntimeError("Active game does not match sampler")
        if verify_history:
            board = arena_replay(identity, active["job"], active["actions"])
            if to_dict(board) != active["state"]:
                raise RuntimeError("Active game history/actions mismatch")
        if len(active["moves"]) != len(active["actions"]):
            raise RuntimeError("Active move audit does not match applied moves")
    return True

def arena_seed(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def arena_commit(run_dir, snapshot, players, *, first=False):
    for player in players.values():
        player.check_frozen()
    snapshot["model"] = {name: p.weights() for name, p in players.items()}
    snapshot["runtime"] = {name: p.runtime() for name, p in players.items()}
    snapshot["rng"] = capture_rng()
    if not first:
        snapshot["global_step"] += 1
    snapshot["metrics"] = {"recorded_games": len(snapshot["results"]),
                           "complete_games": sum(r["status"] == "COMPLETE" for r in snapshot["results"])}
    arena_validate_snapshot(snapshot, snapshot["config"], verify_history=False)
    _save_checkpoint_unlocked(run_dir, snapshot, best=first or snapshot["complete"], resume_cell=ARENA_RESUME)

def arena_finish_game(snapshot, status):
    active = snapshot["active"]
    board = from_dict(active["state"])
    score = area_score(board) if status == "COMPLETE" else None
    candidate_score = (0.5 if score == 0 else float(score * active["job"]["candidate_color"] > 0)) if score is not None else None
    snapshot["results"].append({"job": active["job"], "actions": active["actions"], "moves": active["moves"],
                                "final_state": active["state"], "status": status,
                                "black_score": score, "candidate_score": candidate_score})
    snapshot["sampler"]["next_game"] += 1
    snapshot["active"] = None
    protocol = ArenaProtocol(**snapshot["config"]["protocol"])
    snapshot["complete"] = snapshot["sampler"]["next_game"] == len(arena_jobs(protocol))

def arena_run(project, run_id, protocol, players, *, stop_after_events=None):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,100}", run_id):
        raise ValueError("Use a simple nonempty run ID")
    if stop_after_events is not None and (type(stop_after_events) is not int or stop_after_events < 1):
        raise ValueError("stop_after_events must be a positive integer")
    identity = arena_identity(project, protocol, players)
    estimate = 2 * len(protocol.seeds) * protocol.max_plies * protocol.time_budget_s
    if estimate > 900:
        receipt_path = Path(project) / "reports/11_arena_validation.json"
        receipt = json.loads(receipt_path.read_text()) if receipt_path.exists() else {}
        if receipt.get("exact_resume") is not True or receipt.get("sources") != arena_sources(project):
            raise RuntimeError("Long arena requires a current successful recovery receipt")
    run_dir = Path(project) / "checkpoints" / run_id
    with checkpoint_lock(run_dir):
        snapshot = arena_strict_load(run_dir)
        if snapshot is None:
            if (run_dir / "identity.json").exists() and json.loads((run_dir / "identity.json").read_text()) != identity:
                raise RuntimeError("Run directory already has a different identity")
            atomic_json(run_dir / "identity.json", identity)
            arena_seed(protocol.seeds[0])
            snapshot = {"schema": ARENA_SCHEMA, "identity_sha256": arena_digest(identity), "config": identity,
                        "model": {}, "runtime": {}, "initial_runtime": {name: p.runtime() for name,p in players.items()}, "optimizer": None, "scheduler": None, "scaler": None,
                        "rng": None, "sampler": {"next_game": 0}, "global_step": 0,
                        "metrics": {}, "optimizer_steps": 0, "replay": None, "distributed": {"world_size": 1},
                        "results": [], "active": None, "complete": False}
            arena_commit(run_dir, snapshot, players, first=True)
        else:
            arena_validate_snapshot(snapshot, identity)
            if not (run_dir / "identity.json").is_file() or json.loads((run_dir / "identity.json").read_text()) != identity:
                raise RuntimeError("Committed arena identity missing or changed")
            for name, player in players.items():
                player.restore(snapshot["model"][name], snapshot["runtime"][name])
            restore_rng(snapshot["rng"])
        events = 0
        while not snapshot["complete"]:
            if snapshot["active"] is None:
                job = arena_jobs(protocol)[snapshot["sampler"]["next_game"]]
                arena_seed(job["seed"])
                for name, player in players.items():
                    if player.set_runtime_state:
                        player.set_runtime_state(copy.deepcopy(snapshot["initial_runtime"][name]))
                snapshot["active"] = {"job": job, "state": copy.deepcopy(identity["openings"][job["block"]]["state"]), "actions": [], "moves": []}
                arena_commit(run_dir, snapshot, players)
                events += 1
            else:
                active = snapshot["active"]
                board = from_dict(active["state"])
                if board.terminal:
                    arena_finish_game(snapshot, "COMPLETE")
                elif board.move_number >= protocol.max_plies:
                    arena_finish_game(snapshot, "UNKNOWN_MOVE_CAP")
                else:
                    name = protocol.candidate if board.to_play == active["job"]["candidate_color"] else protocol.opponent
                    player = players[name]
                    player.check_frozen()
                    if player.model is not None and next(player.model.parameters()).device.type == "cuda":
                        torch.cuda.synchronize(next(player.model.parameters()).device)
                    started = time.perf_counter()
                    failure = None
                    try:
                        if player.prepare_position is not None:
                            player.prepare_position(board, copy.deepcopy(identity["openings"][active["job"]["block"]]["actions"]), copy.deepcopy(active["actions"]))
                        with torch.inference_mode():
                            result = player.decide(board, protocol.time_budget_s)
                        if player.model is not None and next(player.model.parameters()).device.type == "cuda":
                            torch.cuda.synchronize(next(player.model.parameters()).device)
                    except Exception as error:
                        result = {"action": None, "exception": type(error).__name__ + ": " + str(error)[:200]}
                        failure = "UNKNOWN_PLAYER_ERROR"
                    elapsed = time.perf_counter() - started
                    player.check_frozen()
                    if not isinstance(result, dict):
                        result = {"action": None}
                        failure = "UNKNOWN_PLAYER_ERROR"
                    action = result.get("action")
                    if elapsed > protocol.time_budget_s or result.get("deadline_missed", False):
                        failure = "UNKNOWN_DEADLINE"
                    counters = result.get("counters", {})
                    for cap, limit in protocol.work_budget.items():
                        counter = {"max_nodes": "nodes", "max_model_calls": "model_calls"}[cap]
                        count = counters.get(counter)
                        if type(count) is not int or not 0 <= count <= limit:
                            failure = "UNKNOWN_WORK_BUDGET"
                    try:
                        successor = play(board, action)
                        if arena_reference_move(board, action) != successor.board:
                            raise RuntimeError("Independent coordinate verifier disagrees with arena transition")
                        legal = True
                    except (IllegalMove, TypeError, ValueError):
                        successor, legal = None, False
                        if failure is None:
                            failure = "UNKNOWN_ILLEGAL_MOVE"
                    audit = {"player": name, "action": action if type(action) is int else None,
                             "legal": legal, "elapsed_s": elapsed, "deadline_missed": elapsed > protocol.time_budget_s or bool(result.get("deadline_missed", False)),
                             "failure": failure, "counters": {k: v for k, v in counters.items() if type(v) in (int, float, str, bool)},
                             "strategic_status": "UNKNOWN", "rng_probe": result.get("rng_probe")}
                    if failure:
                        active["moves"].append(audit)
                        arena_finish_game(snapshot, failure)
                    else:
                        active["moves"].append(audit)
                        active["actions"].append(action)
                        active["state"] = to_dict(successor)
                arena_commit(run_dir, snapshot, players)
                events += 1
            if stop_after_events is not None and events >= stop_after_events:
                break
        summary = arena_summarize(protocol, snapshot["results"], players)
        atomic_json(run_dir / "summary.json", summary)
        return snapshot, summary


## Summarize evidence and ablations

Report wins/draws/losses only for completed games; report every unknown separately. The all-planned score bounds include unfinished games as potentially zero or one, so dropping hard games cannot improve the reported lower bound. The interval is computed on fully paired blocks. Promotion additionally requires *every* planned game and block to be complete.

For AHR ablations, declare settings such as `pheromone_enabled`, Queen/Worker/Scout iteration counts, and fixed `work_budget` inside each player's immutable config. Implement these settings in the actual player factory; the arena does not pretend a label changes model behavior. Run separate predeclared protocols per ablation, retain model/source hashes, equal time limits and equal node/model-call caps, and allocate confidence across comparisons. Fewer recurrence steps change FLOPs even when node counts match; measure GPU time as well before claiming matched compute.

In [ ]:
def arena_summarize(protocol, results, players):
    complete = [row for row in results if row["status"] == "COMPLETE"]
    paired = []
    for block in range(len(protocol.seeds)):
        rows = [r for r in complete if r["job"]["block"] == block]
        if len(rows) == 2 and {r["job"]["candidate_color"] for r in rows} == {BLACK, WHITE}:
            paired.append(sum(r["candidate_score"] for r in rows) / 2)
    mean = sum(paired) / len(paired) if paired else None
    radius = math.sqrt(math.log(2 / protocol.alpha) / (2 * len(paired))) if paired else None
    interval = [max(0., mean-radius), min(1., mean+radius)] if paired else [0., 1.]
    planned = 2 * len(protocol.seeds)
    total = sum(row["candidate_score"] for row in complete)
    audits = [move for row in results for move in row["moves"]]
    failures = sum(move["failure"] is not None for move in audits)
    finished = len(results) == planned
    eligible = (protocol.purpose == "development_strength" and protocol.size == 9 and finished
                and len(complete) == planned and len(paired) >= max(128, protocol.minimum_blocks) and failures == 0)
    promote = bool(eligible and interval[0] > protocol.promotion_threshold)
    return {"schema": ARENA_SCHEMA, "purpose": protocol.purpose,
            "evaluation_status": "NOT_EVALUATED" if protocol.purpose != "development_strength" else ("COMPLETE" if finished else "IN_PROGRESS"),
            "planned_games": planned, "recorded_games": len(results), "completed_games": len(complete),
            "status_counts": dict(Counter(row["status"] for row in results)), "completed_paired_blocks": len(paired),
            "candidate_wins": sum(row["candidate_score"] == 1 for row in complete),
            "candidate_draws": sum(row["candidate_score"] == 0.5 for row in complete),
            "candidate_losses": sum(row["candidate_score"] == 0 for row in complete),
            "paired_mean_score": mean, "paired_hoeffding_interval": interval, "alpha": protocol.alpha,
            "all_planned_score_bounds": [total/planned, (total+planned-len(complete))/planned],
            "illegal_proposals": sum(not m["legal"] for m in audits),
            "deadline_misses": sum(m["deadline_missed"] for m in audits), "failed_move_calls": failures,
            "promotion_eligible": eligible, "promotion_recommended": promote,
            "promotion_threshold": protocol.promotion_threshold, "minimum_blocks": protocol.minimum_blocks,
            "promotion_effect": "Recommendation only; no checkpoint pointer or deployed service is changed.",
            "elo": None, "full_game_optimality": "NOT_ESTABLISHED", "optimizer_steps": 0,
            "latency_by_player": {name: arena_latency([m for m in audits if m["player"] == name], protocol.time_budget_s) for name in players},
            "player_identities": {name: player.identity() for name, player in sorted(players.items())}}

def arena_fixture_player(name, *, mode="recovery"):
    runtime = {"calls_in_game": 0}
    model = torch.nn.Linear(2, 1)
    with torch.no_grad():
        model.weight.fill_(0.25)
        model.bias.zero_()
    def decide(state, budget):
        runtime["calls_in_game"] += 1
        if mode == "illegal":
            action = -1
        elif mode == "timeout":
            return {"action": state.pass_action, "deadline_missed": True}
        elif mode == "pass" or state.move_number >= 4:
            action = state.pass_action
        else:
            choices = [a for a in legal_actions(state) if a != state.pass_action]
            action = random.choice(choices) if choices else state.pass_action
        return {"action": action, "rng_probe": [random.random(), float(np.random.random()), float(torch.rand(()))]
                + [float(torch.rand((), device=f"cuda:{i}").cpu()) for i in range(torch.cuda.device_count())]
                + [runtime["calls_in_game"]]}
    def prepare(state, opening_actions, applied_actions):
        replayed = initial_state(state.size,state.komi)
        for action in opening_actions + applied_actions:
            replayed = arena_reference_child(replayed,action)
        if replayed is None or to_dict(replayed) != to_dict(state):
            raise RuntimeError("Full chronological history preparation mismatch")
    return FrozenArenaPlayer(name, decide, {"mode": mode}, {"kind": "fixture", "purpose": "wiring_only"},
                             model=model, get_runtime_state=lambda: dict(runtime),
                             set_runtime_state=lambda state: runtime.update(copy.deepcopy(state)), prepare_position=prepare)

def arena_semantic_result(snapshot):
    result = copy.deepcopy(snapshot["results"])
    for row in result:
        for move in row["moves"]:
            move.pop("elapsed_s")
    return result

def arena_expect_rejected(function, label):
    try:
        function()
    except (RuntimeError, ValueError, KeyError, TypeError):
        return label
    raise AssertionError("Validation should reject: " + label)


In [ ]:
def arena_go_player(project, name, run_id, *, device="cpu", selection="best", search_config=None, work_budget=None):
    """Load a real 08/09 candidate through the hash-checking 08 loader, lazily."""
    import types, sys
    module_name = "ahr11_go_adapter_" + uuid.uuid4().hex
    module = types.ModuleType(module_name)
    sys.modules[module_name] = module
    namespace = module.__dict__
    namespace.update(PROJECT=Path(project), nbformat=nbformat, Path=Path)
    for number in (*range(7), 8):
        paths = list((Path(project) / "notebooks").glob(f"{number:02d}_*.ipynb"))
        if len(paths) != 1:
            raise RuntimeError(f"Expected one upstream notebook {number:02d}")
        for cell in nbformat.read(paths[0], as_version=4).cells:
            if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", []):
                exec(compile(cell.source, paths[0].name + ":ahr-export", "exec"), namespace)
    model, metadata = namespace["load_go_ahr"](project, run_id, device=device, selection=selection)
    search = namespace["GoAHRSearchConfig"](**(search_config or {}))
    work_budget = dict(work_budget or {})
    if "max_nodes" in work_budget and search.max_nodes != work_budget["max_nodes"]:
        raise ValueError("Search node cap must match the predeclared arena work budget")
    provenance = {"kind": "ahr_checkpoint", "purpose": "go_training",
                  "trained_optimizer_steps": metadata["optimizer_step"],
                  "checkpoint_sha256": metadata["checkpoint_sha256"],
                  "checkpoint_file": metadata["checkpoint_file"],
                  "source_hashes": metadata["source_hashes"],
                  "loader_metadata": {key: value for key, value in metadata.items() if key != "initial_model_state"}}
    def decide(state, budget):
        # Upstream notebooks own a separate GoState class in this private module.
        # Rehydrate the complete history instead of passing a cross-namespace instance.
        native_state = namespace["GoState"].from_dict(to_dict(state))
        result = namespace["choose_go_ahr_move"](native_state, model, time_budget_s=budget, search_config=search)
        return {"action": result["action"], "strategic_status": "UNKNOWN",
                "deadline_missed": result.get("deadline_missed", False),
                "counters": {"nodes": result.get("nodes"), "model_calls": result.get("model_calls")}}
    return FrozenArenaPlayer(name, decide, {"search": asdict(search), "model": asdict(model.config), "work_budget": work_budget}, provenance, model=model)

def arena_fresh_fixture_process(project, run_id, protocol, *, stop_after_events=None):
    """Fresh Python process; all reusable logic remains in notebook export cells."""
    import subprocess, sys
    bootstrap = """import sys,json,nbformat
from pathlib import Path
project=Path(sys.argv[1])
for number in (0,1,11):
    path=next((project/'notebooks').glob(f'{number:02d}_*.ipynb'))
    for cell in nbformat.read(path,as_version=4).cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
            exec(compile(cell.source,path.name+':ahr-export','exec'),globals())
torch.set_num_threads(1)
protocol=ArenaProtocol(**json.loads(sys.argv[3]))
players={name:arena_fixture_player(name) for name in ('candidate','reference')}
limit=None if sys.argv[4]=='none' else int(sys.argv[4])
arena_run(project,sys.argv[2],protocol,players,stop_after_events=limit)
"""
    subprocess.run([sys.executable, "-c", bootstrap, str(project), run_id, arena_json(asdict(protocol)),
                    "none" if stop_after_events is None else str(stop_after_events)],
                   cwd=project, check=True, timeout=90, capture_output=True, text=True)


## V2 development tactics and predeclared generations

The synthetic setup suite contains 32 rotated 9×9 positions. Independent coordinate transitions exhaustively determine one-ply capture, two-ply survival, three-ply forced capture, ko legality and exact terminal-pass predicates. A missed predicate means only that the selected move misses that stated horizon; it is never a proved game loss. Ladder and life-and-death coverage is deliberately bounded. These fixtures are development diagnostics, not training data or a lockbox.

`arena_declare_family(PROJECT, "generation-v2", generations=3, blocks=128)` fixes disjoint opening seeds and a total 0.05 error budget before evaluation. Each candidate is tested once at alpha=0.05/3 against the then-selected frozen champion. All 256 games must complete without a failed call, and the paired-block lower confidence bound must exceed 0.55. The function only recommends promotion; the pipeline records the explicit selection.

`arena_tactical_evaluate(PROJECT, run_id, report_id, device="cpu")` runs a bounded diagnostic and writes `reports/11_v2_<report_id>_tactics.json`. Typical runtime is below two minutes. It contains no training or long-running workflow. `arena_evaluate_generation(PROJECT, family_id, generation, candidate_run, reference_run, device="cuda:1")` resumes exact committed arena state from `checkpoints/arena11-<family_id>-g<NN>/`; use the same inputs and source. The standard fresh-process recovery smoke must pass before long arenas. For an interruption, reload tagged definitions and repeat exactly the same call. External engines are independent evaluation opponents only; because algorithms differ, cross-engine comparisons use a common wall-clock cap and report algorithm-specific work rather than asserting matched FLOPs.


In [ ]:
def arena_reference_move(state, action):
    """Independent coordinate-map implementation; return None if illegal."""
    n = state.size
    if state.consecutive_passes == 2 or action < 0 or action > n * n:
        return None
    if action == n * n:
        return state.board
    grid = {(r, c): state.board[r * n + c] for r in range(n) for c in range(n)}
    location = divmod(action, n)
    if grid[location]:
        return None
    grid[location] = state.to_play

    def adjacent(rc):
        r, c = rc
        return [(x, y) for x, y in ((r, c - 1), (r, c + 1), (r - 1, c), (r + 1, c)) if (x, y) in grid]

    def component(origin):
        color = grid[origin]
        found = {origin}
        frontier = [origin]
        liberty = False
        for point in frontier:
            for other in adjacent(point):
                if grid[other] == 0:
                    liberty = True
                if grid[other] == color and other not in found:
                    found.add(other)
                    frontier.append(other)
        return found, liberty

    # Determine all opposing dead groups before removing any, independently of
    # the production implementation's neighbor-by-neighbor removal order.
    captured = set()
    for other in adjacent(location):
        if grid[other] == -state.to_play:
            stones, has_liberty = component(other)
            if not has_liberty:
                captured |= stones
    for stone in captured:
        grid[stone] = 0
    _, has_liberty = component(location)
    if not has_liberty:
        return None
    result = tuple(grid[(r, c)] for r in range(n) for c in range(n))
    return None if result in state.history else result


def arena_reference_score(state):
    """Independent per-empty-point reachability scoring (no shared helpers)."""
    n = state.size
    total = -state.komi
    for start, occupant in enumerate(state.board):
        if occupant:
            total += occupant
            continue
        discovered = {start}
        frontier = [start]
        sees_black = False
        sees_white = False
        for current in frontier:
            row, col = divmod(current, n)
            for rr, cc in ((row - 1, col), (row + 1, col), (row, col - 1), (row, col + 1)):
                if not (0 <= rr < n and 0 <= cc < n):
                    continue
                other = rr * n + cc
                color = state.board[other]
                sees_black |= color == 1
                sees_white |= color == -1
                if color == 0 and other not in discovered:
                    discovered.add(other)
                    frontier.append(other)
        if sees_black and not sees_white:
            total += 1
        if sees_white and not sees_black:
            total -= 1
    return float(total)

def arena_latency(audits, budget):
    values = np.asarray([float(a["elapsed_s"]) for a in audits], dtype=float)
    if not len(values):
        return {"calls": 0, "p50_s": None, "p95_s": None, "p99_s": None, "max_s": None,
                "budget_s": budget, "deadline_misses": 0, "timing_scope": "complete decision including position reconstruction"}
    return {"calls": int(len(values)), "p50_s": float(np.quantile(values, .50)),
            "p95_s": float(np.quantile(values, .95)), "p99_s": float(np.quantile(values, .99)),
            "max_s": float(values.max()), "mean_s": float(values.mean()), "budget_s": budget,
            "deadline_misses": sum(bool(a.get("deadline_missed", a["elapsed_s"] > budget)) for a in audits),
            "timing_scope": "complete decision including position reconstruction; checkpoint I/O excluded"}


def arena_reference_child(state, action):
    """Independent board transition plus explicit state bookkeeping."""
    board = arena_reference_move(state, action)
    if board is None:
        return None
    is_pass = action == state.pass_action
    return GoState(state.size, board, to_play=-state.to_play, komi=state.komi,
                   history=state.history | {board},
                   consecutive_passes=state.consecutive_passes + 1 if is_pass else 0,
                   move_number=state.move_number + 1)


def arena_reference_group(board, size, anchor):
    """Independent coordinate flood fill, separate from production group helpers."""
    color = board[anchor]
    if color == EMPTY:
        return set(), set()
    group, liberties, todo = {anchor}, set(), [anchor]
    while todo:
        point = todo.pop()
        row, col = divmod(point, size)
        for rr, cc in ((row-1,col),(row+1,col),(row,col-1),(row,col+1)):
            if 0 <= rr < size and 0 <= cc < size:
                other = rr*size + cc
                if board[other] == EMPTY:
                    liberties.add(other)
                elif board[other] == color and other not in group:
                    group.add(other); todo.append(other)
    return group, liberties


def arena_tactical_success(case, action):
    """Exact stated bounded predicate, never a full-game value or learned teacher."""
    state = from_dict(case["state"])
    child = arena_reference_child(state, action)
    if child is None:
        return False
    goal = case["goal"]
    targets = case.get("targets", [])
    if goal == "legal":
        return True
    if goal == "capture_one_ply":
        return all(child.board[p] != state.board[p] for p in targets)
    if goal == "terminal_pass_score":
        return action == state.pass_action and child.terminal and arena_reference_score(child) == case["terminal_black_score"]
    if goal == "survive_two_plies":
        color = state.board[targets[0]]
        if any(child.board[p] != color for p in targets):
            return False
        for reply in range(child.pass_action + 1):
            after = arena_reference_child(child, reply)
            if after is not None and any(after.board[p] != color for p in targets):
                return False
        return True
    if goal == "force_capture_three_plies":
        color = state.board[targets[0]]
        if all(child.board[p] != color for p in targets):
            return True
        anchor = next(p for p in targets if child.board[p] == color)
        _, liberties = arena_reference_group(child.board, child.size, anchor)
        # Defender can pass. With >1 liberty the attacker cannot capture in the
        # sole remaining move, so this is an exact counterexample at horizon 3.
        if len(liberties) != 1:
            return False
        for reply in range(child.pass_action + 1):
            after = arena_reference_child(child, reply)
            if after is None:
                continue
            surviving = [p for p in targets if after.board[p] == color]
            if not surviving:
                continue
            _, response_liberties = arena_reference_group(after.board, after.size, surviving[0])
            if len(response_liberties) != 1:
                return False
            capture = arena_reference_child(after, next(iter(response_liberties)))
            if capture is None or any(capture.board[p] == color for p in targets):
                return False
        return True
    raise ValueError("Unknown bounded tactical predicate")


def arena_tactical_suite():
    """Hand-authored 9x9 setup positions; four rotations, development use only."""
    bases = []
    def setup(black, white, to_play=BLACK, passes=0):
        board = [EMPTY]*81
        for points, color in ((black, BLACK), (white, WHITE)):
            for point in points:
                if board[point] != EMPTY:
                    raise ValueError("Overlapping setup stones")
                board[point] = color
        return GoState(9, tuple(board), to_play=to_play, consecutive_passes=passes)
    def add(name, category, state, goal, targets=(), **extra):
        bases.append({"name": name, "category": category, "state": to_dict(state),
                      "goal": goal, "targets": list(targets), **extra})
    add("capture_single", "capture", setup([1,9,19], [10]), "capture_one_ply", [10])
    add("capture_connected", "capture", setup([2,9], [0,1]), "capture_one_ply", [0,1])
    add("escape_atari", "atari_escape", setup([10], [1,9,19]), "survive_two_plies", [10])
    ko = setup([1,9,19], [10,2,12,20])
    add("take_ko", "ko", ko, "capture_one_ply", [10])
    ko_after = arena_reference_child(ko, 11)
    assert ko_after is not None and arena_reference_move(ko_after, 10) is None
    add("respect_ko_history", "ko", ko_after, "legal", forbidden=[10])
    # A one-eye corner group: filling its last eye captures all five stones.
    add("kill_one_eye", "life_death_bounded", setup([3,12,18,19,20], [0,2,9,10,11]),
        "capture_one_ply", [0,2,9,10,11])
    # A short ladder-shaped corner finish. Every legal defender reply is tested;
    # failure means only 'not forced inside three plies', never a proved survival.
    add("short_ladder_finish", "ladder_bounded", setup([1,9,18,20,28], [10]),
        "force_capture_three_plies", [10])
    forced = setup([a for a in range(81) if a not in (0,8)], [], to_play=WHITE, passes=1)
    add("forced_final_pass", "passing_endgame", forced, "terminal_pass_score",
        terminal_black_score=73.5)
    cases = []
    def point(p, rotation):
        if p == 81:
            return p
        row, col = divmod(p, 9)
        for _ in range(rotation):
            row, col = col, 8-row
        return row*9+col
    def board_rotate(board, rotation):
        result = [0]*81
        for p,c in enumerate(board):
            result[point(p, rotation)] = c
        return tuple(result)
    for base in bases:
        for rotation in range(4):
            original = from_dict(base["state"])
            state = GoState(9, board_rotate(original.board, rotation), to_play=original.to_play,
                komi=original.komi, history=frozenset(board_rotate(b, rotation) for b in original.history),
                consecutive_passes=original.consecutive_passes, move_number=original.move_number)
            case = copy.deepcopy(base)
            case.update(name=base["name"]+f"_r{rotation}", state=to_dict(state),
                        targets=[point(p,rotation) for p in base["targets"]],
                        fixture_origin="synthetic setup; history membership explicit; empty-board reachability not asserted",
                        scope="Exact stated predicate at horizon at most three plies; full-game result UNKNOWN")
            if "forbidden" in case:
                case["forbidden"] = [point(p, rotation) for p in case["forbidden"]]
            expected = []
            for action in range(82):
                reference = arena_reference_child(state, action)
                try:
                    actual = play(state, action)
                except IllegalMove:
                    actual = None
                if (actual is None) != (reference is None) or actual is not None and to_dict(actual) != to_dict(reference):
                    raise AssertionError("Independent tactical transition disagreement")
                if arena_tactical_success(case, action):
                    expected.append(action)
            if not expected:
                raise AssertionError("Tactical fixture has no satisfying move: " + case["name"])
            if any(p in expected for p in case.get("forbidden", [])):
                raise AssertionError("Ko fixture admitted forbidden recapture")
            case["expected_actions"] = expected
            case["predicate_status"] = "EXHAUSTIVELY_VERIFIED_BOUNDED"
            cases.append(case)
    return {"schema": "ahr11-tactical-development-v2", "rules": RULESET, "cases": cases,
            "lockbox": False, "training_use": "NONE: evaluation-only hand-authored constraints",
            "claims": "No full-game optimality, rank, Elo, or complete ladder/life-and-death solver claim"}


def arena_tactical_evaluate(project, run_id, report_id, *, device="cpu", selection="latest",
                            time_budget_s=1.0, search_config=None):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,100}", report_id):
        raise ValueError("Use a simple report ID")
    suite = arena_tactical_suite()
    search_config = search_config or {"max_depth": 2, "branch_width": 4, "max_nodes": 32, "queen_cycles": 1}
    player = arena_go_player(project, "candidate", run_id, device=device, selection=selection, search_config=search_config)
    identity = {"suite_sha256": arena_digest(suite), "player": player.identity(), "sources": arena_sources(project),
                "time_budget_s": time_budget_s, "device": device, "selection": selection}
    path = Path(project)/"reports"/("11_v2_"+report_id+"_tactics.json")
    if path.exists():
        existing = json.loads(path.read_text())
        if existing.get("identity") != identity:
            raise RuntimeError("Tactical report identity changed; use a new report ID")
        if existing.get("status") == "COMPLETE":
            return existing
    rows = []
    for case in suite["cases"]:
        state = from_dict(case["state"])
        arena_seed(int(arena_digest(case)[:8], 16))
        if player.model is not None and next(player.model.parameters()).device.type == "cuda":
            torch.cuda.synchronize(next(player.model.parameters()).device)
        started = time.perf_counter()
        failure = None
        try:
            with torch.inference_mode():
                result = player.decide(state, time_budget_s)
            if player.model is not None and next(player.model.parameters()).device.type == "cuda":
                torch.cuda.synchronize(next(player.model.parameters()).device)
        except Exception as error:
            result = {"action": None, "exception": str(error)[:200]}
            failure = "UNKNOWN_PLAYER_ERROR"
        elapsed = time.perf_counter() - started
        action = result.get("action")
        legal = type(action) is int and 0 <= action <= 81 and arena_reference_move(state, action) is not None
        deadline = elapsed > time_budget_s or bool(result.get("deadline_missed",False))
        if deadline:
            failure = "UNKNOWN_DEADLINE"
        elif not legal:
            failure = failure or "UNKNOWN_ILLEGAL_MOVE"
        status = failure or ("SATISFIED_BOUNDED_PREDICATE" if action in case["expected_actions"] else "NOT_SATISFIED_BOUNDED_PREDICATE")
        rows.append({"case": case["name"], "category": case["category"], "action": action,
                     "status": status, "legal": legal, "elapsed_s": elapsed, "deadline_missed": deadline,
                     "expected_actions": case["expected_actions"], "strategic_result": "UNKNOWN",
                     "counters": result.get("counters",{})})
        player.check_frozen()
    categories = {}
    for category in sorted({r["category"] for r in rows}):
        group = [r for r in rows if r["category"] == category]
        categories[category] = {"cases": len(group), "status_counts": dict(Counter(r["status"] for r in group))}
    report = {"schema": "ahr11-tactical-evaluation-v2", "status": "COMPLETE", "identity": identity,
              "suite": suite, "results": rows, "category_summary": categories,
              "status_counts": dict(Counter(r["status"] for r in rows)),
              "latency": arena_latency(rows,time_budget_s), "promotion_effect": "NONE", "general_strength": "NOT_ESTABLISHED"}
    atomic_json(path, report)
    return report


def arena_declare_family(project, family_id, *, generations=3, blocks=128, seed=2026092411,
                         time_budget_s=1.0, max_plies=256, search_config=None):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,70}", family_id):
        raise ValueError("Use a simple family ID of at most 71 characters")
    if not 1 <= generations <= 20 or blocks < 128:
        raise ValueError("Predeclare 1-20 generations and at least 128 paired blocks each")
    search_config = search_config or {"max_depth": 2, "branch_width": 4, "max_nodes": 32, "queen_cycles": 1}
    rng = random.Random(seed)
    seeds = rng.sample(range(1,2**31), generations*blocks)
    plan = {"schema": "ahr11-evaluation-family-v2", "family_id": family_id, "generations": generations,
            "paired_blocks_per_generation": blocks, "seed": seed, "family_alpha": .05,
            "alpha_per_generation": .05/generations, "time_budget_s": time_budget_s,
            "max_plies": max_plies, "search_config": search_config,
            "generation_seeds": [seeds[i*blocks:(i+1)*blocks] for i in range(generations)],
            "promotion_threshold": .55, "minimum_blocks": 128,
            "multiple_testing": "Bonferroni allocation fixed before play; fresh disjoint opening seeds per generation; no optional stopping",
            "reference_selection": "Previously selected champion, frozen before this generation's evaluation",
            "rules": RULESET, "sources": arena_sources(project)}
    path = Path(project)/"reports"/("11_v2_family_"+family_id+".json")
    with checkpoint_lock(Path(project)/"checkpoints"/("arena11-family-"+family_id)):
        if path.exists():
            if json.loads(path.read_text()) != plan:
                raise RuntimeError("Predeclared evaluation family changed")
        else:
            atomic_json(path, plan)
    return plan


def arena_evaluate_generation(project, family_id, generation, candidate_run, reference_run, *,
                              device="cpu", stop_after_events=None):
    path = Path(project)/"reports"/("11_v2_family_"+family_id+".json")
    plan = json.loads(path.read_text())
    if type(generation) is not int or not 1 <= generation <= plan["generations"]:
        raise ValueError("Generation is outside the predeclared family")
    if plan["sources"] != arena_sources(project):
        raise RuntimeError("Evaluation source changed after family declaration")
    work = {"max_nodes": plan["search_config"]["max_nodes"]}
    players = {name: arena_go_player(project,name,run,device=device,selection="latest",
                search_config=plan["search_config"],work_budget=work)
               for name,run in (("candidate",candidate_run),("reference",reference_run))}
    # Hash-verified loader metadata supplies each immutable source file. Check
    # training completion under its writer lock before freezing any candidate.
    ready = {}
    for name, run in (("candidate",candidate_run),("reference",reference_run)):
        directory = Path(project)/"checkpoints"/run
        with checkpoint_lock(directory):
            checkpoint_file = players[name].provenance["checkpoint_file"]
            checkpoint_path = Path(checkpoint_file)
            if not checkpoint_path.is_absolute():
                checkpoint_path = directory/checkpoint_path.name
            if hashlib.sha256(checkpoint_path.read_bytes()).hexdigest() != players[name].provenance["checkpoint_sha256"]:
                raise RuntimeError("Frozen training input changed")
            checkpoint = torch.load(checkpoint_path,map_location="cpu",weights_only=False)
            ready[name] = {"phase": checkpoint.get("phase"), "optimizer_steps": checkpoint.get("optimizer_step",0),
                           "checkpoint_sha256": players[name].provenance["checkpoint_sha256"]}
            if checkpoint.get("phase") != "complete" or checkpoint.get("optimizer_step",0) < 1:
                raise RuntimeError("Generation comparison needs two completed trained self-play runs")
    protocol = ArenaProtocol(seeds=tuple(plan["generation_seeds"][generation-1]),
        purpose="development_strength", alpha=plan["alpha_per_generation"], minimum_blocks=128,
        promotion_threshold=plan["promotion_threshold"], time_budget_s=plan["time_budget_s"],
        max_plies=plan["max_plies"],work_budget=work,
        error_budget_note=plan["multiple_testing"]+f"; family={family_id}; generation={generation}")
    run_id = f"arena11-{family_id}-g{generation:02d}"
    request = {"family_sha256":arena_digest(plan),"generation":generation,"candidate_run":candidate_run,
               "reference_run":reference_run,"device":device,"identity":arena_identity(project,protocol,players),"readiness":ready}
    request_path = Path(project)/"reports"/("11_v2_"+family_id+f"_g{generation:02d}_request.json")
    if request_path.exists() and json.loads(request_path.read_text()) != request:
        raise RuntimeError("Generation comparison inputs changed")
    if not request_path.exists():
        atomic_json(request_path,request)
    snapshot,summary = arena_run(project,run_id,protocol,players,stop_after_events=stop_after_events)
    atomic_json(Path(project)/"reports"/("11_v2_"+family_id+f"_g{generation:02d}_results.json"),
                {"request":request,"summary":summary,"complete":snapshot["complete"],"automatic_promotion":False})
    return snapshot,summary


## Run short wiring, recovery and failure checks

The fixture players deliberately pass after a few moves. Their results say nothing about Go strength. We compare uninterrupted versus interrupted/resumed move histories and RNG probes, then verify rejection of corrupt manifests, changed protocols, changed weights and malformed history. Deadline, illegal-move and move-cap tests must retain UNKNOWN. These checks train no model.

When `go08-initial-warmup-v2` exists (override with `AHR11_WIRING_RUN`), a second short check loads that actual Go-trained model through notebook 08. The same frozen checkpoint plays both colours for two additional plies after a two-ply opening, then the cap deliberately records UNKNOWN. This exercises the full-history bridge between notebook namespaces and confirms unchanged weights; it is not a strength comparison. Missing checkpoints are reported as a skipped integration, not silently fabricated.

In [ ]:
sources = arena_sources(PROJECT)
prefix = "arena11-validation-" + arena_digest(sources)[:12]
players = {name: arena_fixture_player(name) for name in ("candidate", "reference")}
protocol = ArenaProtocol(seeds=(1103, 2207), opening_plies=2, max_plies=10, time_budget_s=0.5)
full, full_summary = arena_run(PROJECT, prefix + "-full", protocol, players)
resumed_players = {name: arena_fixture_player(name) for name in ("candidate", "reference")}
partial, _ = arena_run(PROJECT, prefix + "-resume", protocol, resumed_players, stop_after_events=3)
resumed, resumed_summary = arena_run(PROJECT, prefix + "-resume", protocol, resumed_players)
assert arena_semantic_result(full) == arena_semantic_result(resumed)
assert full["sampler"] == resumed["sampler"] and full["global_step"] == resumed["global_step"]
assert full_summary["completed_games"] == 4 and not full_summary["promotion_recommended"]
arena_fresh_fixture_process(PROJECT, prefix + "-fresh", protocol, stop_after_events=3)
arena_fresh_fixture_process(PROJECT, prefix + "-fresh", protocol)
fresh = arena_strict_load(PROJECT / "checkpoints" / (prefix + "-fresh"))
assert arena_semantic_result(full) == arena_semantic_result(fresh)
assert fresh["sampler"] == full["sampler"] and fresh["global_step"] == full["global_step"]
identity = full["config"]
rejections = []
changed = copy.deepcopy(full); changed["sampler"]["next_game"] -= 1
rejections.append(arena_expect_rejected(lambda: arena_validate_snapshot(changed, identity), "cursor"))
changed = copy.deepcopy(full); changed["results"][0]["final_state"]["history"] = []
rejections.append(arena_expect_rejected(lambda: arena_validate_snapshot(changed, identity), "history"))
changed_identity = copy.deepcopy(identity); changed_identity["protocol"]["komi"] += 1
rejections.append(arena_expect_rejected(lambda: arena_validate_snapshot(full, changed_identity), "protocol"))
changed = copy.deepcopy(full); changed["results"][0]["candidate_score"] = 0.25
rejections.append(arena_expect_rejected(lambda: arena_validate_snapshot(changed, identity), "invented result"))
changed = copy.deepcopy(full); changed["optimizer"] = {}
rejections.append(arena_expect_rejected(lambda: arena_validate_snapshot(changed, identity), "optimizer"))
changed = copy.deepcopy(full); changed["model"]["candidate"] = {"extra": torch.ones(1)}
rejections.append(arena_expect_rejected(lambda: arena_validate_snapshot(changed, identity), "weights"))
# Copy only manifests for adversarial rejection; never mutate valid checkpoints.
fault_dir = PROJECT / "checkpoints" / (prefix + "-manifest-fault")
fault_dir.mkdir(parents=True, exist_ok=True)
atomic_json(fault_dir / "manifest.json", {"entries": [], "latest": "missing", "best": "missing"})
rejections.append(arena_expect_rejected(lambda: arena_strict_load(fault_dir), "invalid manifest"))
# A committed-file hash failure must not silently fall back to an older valid state.
hash_fault_dir = PROJECT / "checkpoints" / (prefix + "-hash-fault")
hash_fault_dir.mkdir(parents=True, exist_ok=True)
atomic_write_bytes(hash_fault_dir / "bad.pt", b"corrupted committed payload")
atomic_json(hash_fault_dir / "manifest.json", {"entries": [{"file": "bad.pt", "sha256": "0"*64, "global_step": 1}], "latest": "bad.pt", "best": "bad.pt"})
rejections.append(arena_expect_rejected(lambda: arena_strict_load(hash_fault_dir), "payload hash"))
missing_dir = PROJECT / "checkpoints" / (prefix + "-missing-manifest")
missing_dir.mkdir(parents=True, exist_ok=True)
atomic_write_bytes(missing_dir / "orphan.pt", b"synthetic corruption fixture")
rejections.append(arena_expect_rejected(lambda: arena_strict_load(missing_dir), "missing manifest"))
small_protocol = ArenaProtocol(seeds=(337,), size=3, komi=0.5, opening_plies=0, max_plies=8, time_budget_s=0.5)
small, small_summary = arena_run(PROJECT, prefix + "-small", small_protocol, {name: arena_fixture_player(name, mode="pass") for name in players})
assert small_summary["completed_games"] == 2
unknown_checks = {}
for mode, expected in (("illegal", "UNKNOWN_ILLEGAL_MOVE"), ("timeout", "UNKNOWN_DEADLINE")):
    tested = {"candidate": arena_fixture_player("candidate", mode=mode), "reference": arena_fixture_player("reference", mode="pass")}
    snap, summary = arena_run(PROJECT, prefix + "-" + mode, small_protocol, tested)
    assert summary["status_counts"] == {expected: 2}
    assert all(r["candidate_score"] is None for r in snap["results"])
    assert not summary["promotion_recommended"]
    unknown_checks[mode] = expected
capped_protocol = ArenaProtocol(seeds=(339,), opening_plies=0, max_plies=1, time_budget_s=0.5)
capped, capped_summary = arena_run(PROJECT, prefix + "-cap", capped_protocol, {name: arena_fixture_player(name) for name in players})
assert capped_summary["status_counts"] == {"UNKNOWN_MOVE_CAP": 2}
unknown_checks["cap"] = "UNKNOWN_MOVE_CAP"
# Weight mutation is a hard integrity error, never a quietly scored game.
model_fixture = torch.nn.Linear(2, 1)
frozen_fixture = FrozenArenaPlayer("candidate", lambda s,t: {"action": s.pass_action}, {}, {"kind": "fixture"}, model=model_fixture)
with torch.no_grad():
    model_fixture.weight.add_(1)
rejections.append(arena_expect_rejected(frozen_fixture.check_frozen, "frozen mutation"))
# Real Go checkpoint integration: same checkpoint on both sides, bounded wiring only.
actual_run = os.environ.get("AHR11_WIRING_RUN", "go08-initial-warmup-v2")
actual_integration = {"status": "SKIPPED_NO_GO_CHECKPOINT", "run_id": actual_run, "strength_claim": False}
if (PROJECT / "checkpoints" / actual_run / "manifest.json").is_file():
    device = os.environ.get("AHR11_WIRING_DEVICE", "cpu")
    work = {"max_nodes": 4, "max_model_calls": 4}
    actual_players = {name: arena_go_player(PROJECT, name, actual_run, device=device,
                      search_config={"max_depth": 1, "branch_width": 2, "max_nodes": 4, "queen_cycles": 1},
                      work_budget=work) for name in ("candidate", "reference")}
    before_weights = {name: p.weights_digest() for name,p in actual_players.items()}
    actual_protocol = ArenaProtocol(seeds=(401,), opening_plies=2, max_plies=4, time_budget_s=2.0, work_budget=work)
    actual_id = prefix + "-native-" + actual_players["candidate"].provenance["checkpoint_sha256"][:8]
    actual_state, actual_summary = arena_run(PROJECT, actual_id, actual_protocol, actual_players)
    resumed_actual_players = {name: arena_go_player(PROJECT, name, actual_run, device=device,
                      search_config={"max_depth": 1, "branch_width": 2, "max_nodes": 4, "queen_cycles": 1},
                      work_budget=work) for name in ("candidate", "reference")}
    arena_run(PROJECT, actual_id+"-resume", actual_protocol, resumed_actual_players, stop_after_events=2)
    actual_resumed,_ = arena_run(PROJECT,actual_id+"-resume",actual_protocol,resumed_actual_players)
    assert arena_semantic_result(actual_state) == arena_semantic_result(actual_resumed)
    assert actual_state["global_step"] == actual_resumed["global_step"]
    assert actual_summary["status_counts"] == {"UNKNOWN_MOVE_CAP": 2}
    assert actual_summary["illegal_proposals"] == actual_summary["failed_move_calls"] == 0
    assert all(len(o["state"]["history"]) == 3 for o in actual_state["config"]["openings"])
    assert before_weights == {name: p.weights_digest() for name,p in actual_players.items()}
    assert not actual_summary["promotion_recommended"] and actual_summary["evaluation_status"] == "NOT_EVALUATED"
    actual_integration = {"status": "PASS", "run_id": actual_run, "arena_run": actual_id,
                          "checkpoint_sha256": actual_players["candidate"].provenance["checkpoint_sha256"],
                          "trained_optimizer_steps": actual_players["candidate"].provenance["trained_optimizer_steps"],
                          "status_counts": actual_summary["status_counts"], "cross_namespace_history_preserved": True,
                          "weights_unchanged": True, "same_checkpoint_both_sides": True, "device": device, "strength_claim": False,
                          "actual_model_interrupt_resume": True, "resume_scope": "fixed-work bounded actions/history/eventcursor; elapsed time excluded"}
validation = {"schema": ARENA_SCHEMA, "status": "PASS", "sources": sources,
              "exact_resume": True, "fresh_process_resume": True, "stateful_player_resume": True, "frozen_model_resume": True, "cuda_rng_devices_checked": torch.cuda.device_count(), "exact_resume_scope": "deterministic fixture actions, history, all recorded RNG probes and event cursor; elapsed time excluded",
              "board_sizes": [3,9], "nine_by_nine_completed_fixture_games": full_summary["completed_games"],
              "rejected_faults": rejections, "unknown_checks": unknown_checks,
              "strength_status": "NOT_EVALUATED", "promotion_recommended": False, "optimizer_steps": 0,
              "actual_go_checkpoint_integration": actual_integration,
              "checkpoint_run": prefix + "-resume", "final_or_lockbox_read": False,
              "visual_review": "HTML preview generated separately; no graphical viewer available"}
atomic_json(PROJECT / "reports/11_arena_validation.json", validation)
print(json.dumps(validation, indent=2))
# V2 diagnostics exercise the independent bounded oracle without training.
tactical_suite = arena_tactical_suite()
assert len(tactical_suite["cases"]) == 32
assert {c["category"] for c in tactical_suite["cases"]} == {"capture", "atari_escape", "ko", "life_death_bounded", "ladder_bounded", "passing_endgame"}
assert all(c["expected_actions"] for c in tactical_suite["cases"])
latency_probe = arena_latency([{"elapsed_s": .1}, {"elapsed_s": .3}], .2)
assert latency_probe["p50_s"] == .2 and latency_probe["deadline_misses"] == 1
validation["tactical_cases"] = len(tactical_suite["cases"])
validation["tactical_suite_sha256"] = arena_digest(tactical_suite)
validation["independent_transition_verifier"] = True
validation["latency_summary"] = "PASS"
atomic_json(PROJECT / "reports/11_arena_validation.json", validation)
print(json.dumps({"v2_tactics": len(tactical_suite["cases"]), "status": "PASS"}))

from dataclasses import replace
relaxed_protocol = replace(protocol,purpose="development_strength",minimum_blocks=2)
assert not arena_summarize(relaxed_protocol,full["results"],players)["promotion_eligible"]
validation["minimum_128_pairs_not_relaxable"] = True
validation["full_chronological_history_hook"] = "PASS"
atomic_json(PROJECT / "reports/11_arena_validation.json", validation)


## Run a real arena when candidates are ready

Construct each player with an actual frozen model and a decision callable. Record its checkpoint hash, training step, architecture/source hashes and exact search configuration in `provenance` and `config`. A strength candidate requires `purpose="go_training"` and a positive `trained_optimizer_steps`; both players require immutable checkpoint or engine-binary hashes. Set `ArenaProtocol(purpose="development_strength", seeds=..., minimum_blocks=128, ...)` with independently sampled seeds fixed before play. Call `arena_run` with a new descriptive run ID. Reuse exactly that ID, protocol and player factory to resume.

An external engine or GTP adapter is optional and must preserve its own runtime/random state through the provided hooks, or declare and implement stateless per-position reconstruction. Include the full opening/move history; a board image alone loses superko information. An engine executable hash identifies the opponent; it is not an AHR training teacher.

**Current result:** the default checks establish evaluation wiring, 9×9 legality/scoring integration and interruption recovery. They do not establish improved playing strength. No candidate is promoted and no Elo estimate is reported. Checkpoint backups remain on the persistent workspace disk; before deleting the instance, copy the final/best artifacts off-instance and verify their hashes.